%pip install optbinning pandas xlrd duckdb
%pip install --upgrade pip


In [1]:
import pandas as pd
from itertools import combinations
from optbinning import OptimalBinning


In [2]:
data = pd.read_csv(r"/workspaces/Strategic_Segment_Builder/clean_dataset.csv")
# data = data.drop(columns=["id"])
target = "Approved"
data.head()


,Gender,Age,Debt,Married,BankCustomer,Industry,Ethnicity,YearsEmployed,PriorDefault,Employed,CreditScore,DriversLicense,Citizen,ZipCode,Income,Approved
0,1,30.83,0.000,1,1,Industrials,White,1.25,1,1,1,0,ByBirth,202,0,1
1,0,58.67,4.460,1,1,Materials,Black,3.04,1,1,6,0,ByBirth,43,560,1
2,0,24.50,0.500,1,1,Materials,Black,1.50,1,0,0,0,ByBirth,280,824,1
3,1,27.83,1.540,1,1,Industrials,White,3.75,1,1,5,1,ByBirth,100,3,1
4,1,20.17,5.625,1,1,Industrials,White,1.71,1,0,0,0,ByOtherMeans,120,0,1


In [3]:
data['Approved'].value_counts(normalize=True)*100

Approved
0    55.507246
1    44.492754
Name: proportion, dtype: float64

In [13]:
def target_value_counts(df, target_col, normalize=True, pct=True):
    counts = df[target_col].value_counts(normalize=normalize)
    return counts * 100 if pct else counts

def compute_iv_ranking(df, target, solver="cp"):
    results = []
    for col in df.columns:
        if col == target:
            continue
        try:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category","str"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype, solver=solver)
            optb.fit(df[col].values, df[target].values)
            iv = optb.binning_table.build().IV.iloc[-1] 
            results.append({"variable": col, "iv": iv * 100})
        except Exception as exc:
            print(f"Skipping {col}: {exc}")
    return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

def create_binned_df(df, target, variables):
    binned_df = pd.DataFrame(index=df.index)
    for col in variables:
        dtype = "categorical" if str(df[col].dtype) in ["object", "category","str"] else "numerical"
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(df[col].values, df[target].values)
        binned_df[col] = optb.transform(df[col], metric="bins")
    binned_df[target] = df[target].values
    return binned_df


In [14]:
def one_way_summary(binned_df, target, base_rate, pct=True):
    results = []
    for col in binned_df.columns:
        if col == target:
            continue
        summary = (
            binned_df
            .groupby(col)
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = col + "=" + summary[col].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def two_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2 in combinations(cols, 2):
        summary = (
            binned_df
            .groupby([c1, c2])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = c1 + "=" + summary[c1].astype(str) + sep + c2 + "=" + summary[c2].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def three_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2, c3 in combinations(cols, 3):
        summary = (
            binned_df
            .groupby([c1, c2, c3])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = (
            c1 + "=" + summary[c1].astype(str) + sep +
            c2 + "=" + summary[c2].astype(str) + sep +
            c3 + "=" + summary[c3].astype(str)
        )
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def shortlist_rules(df, min_sample_size=1000, min_lift=2.0, base_rate=None):
    if base_rate is None:
        raise ValueError("base_rate is required")
    threshold = base_rate * 100
    return df.assign(
        shortlist=(
            (df["count"] >= min_sample_size) &
            (df["lift"] >= min_lift) &
            (df["rate"] > threshold)
        ).astype(int)
    )


In [15]:
data.shape

(690, 16)

In [27]:
iv_ranking = compute_iv_ranking(data, target)
top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(data, target, top_vars)
base_rate = data[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=100, min_lift=1.5, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=100, min_lift=1.5, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=100, min_lift=1.5, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PriorDefault=[0.50, inf) & ZipCode=(-inf, 71.5...",109,93.577982,2.103218,1
1,"PriorDefault=[0.50, inf) & ZipCode=(-inf, 71.50)",113,92.920354,2.088438,1
2,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",119,92.436975,2.077574,1
3,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",193,92.227979,2.072876,1
4,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",193,92.227979,2.072876,1
5,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",137,91.240876,2.050691,1
6,"PriorDefault=[0.50, inf) & Employed=[0.50, inf)",228,90.789474,2.040545,1
7,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",152,90.789474,2.040545,1
8,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",227,90.748899,2.039633,1
9,"PriorDefault=[0.50, inf) & Employed=[0.50, inf...",109,88.990826,2.000120,1


In [29]:
def parse_rule(rule_str):
    parts = [p.strip() for p in rule_str.split("&")]
    conditions = []
    for part in parts:
        column, interval = part.split("=", 1)
        column = column.strip()
        interval = interval.strip()
        include_lower = interval.startswith("[")
        include_upper = interval.endswith("]")
        lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
        def to_bound(value):
            value = value.lower()
            if value in {"inf", "infty", "+inf", "+infty"}:
                return float("inf")
            if value in {"-inf", "-infty"}:
                return float("-inf")
            return float(value)
        conditions.append({
            "column": column,
            "lower": to_bound(lower_str),
            "upper": to_bound(upper_str),
            "include_lower": include_lower,
            "include_upper": include_upper,
        })
    return conditions

def build_mask(df, conditions):
    mask = pd.Series(True, index=df.index)
    for cond in conditions:
        column = cond["column"]
        column_values = pd.to_numeric(df[column], errors="coerce")
        current = pd.Series(True, index=df.index)
        if cond["include_lower"]:
            current &= column_values >= cond["lower"]
        else:
            current &= column_values > cond["lower"]
        if cond["include_upper"]:
            current &= column_values <= cond["upper"]
        else:
            current &= column_values < cond["upper"]
        mask &= current.fillna(False)
    return mask

first_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(first_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100


ValueError: could not convert string to float: "stringarray>\n['bybirth'"

In [32]:
first_rule 


"PriorDefault=[0.50, inf) & ZipCode=(-inf, 71.50) & Citizen=<StringArray>\n['ByBirth', 'Temporary']\nLength: 2, dtype: str"

In [12]:
df_filtered.shape

(523065, 30)

In [13]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"V14=(-inf, -1.02) & V12=(-inf, -0.87) & V24=[-...",27629,100.0,2.190839,1
1,"V12=(-inf, -0.87) & V11=[0.91, inf) & V24=[-1....",26211,100.0,2.190839,1
2,"V14=(-inf, -1.02) & V10=(-inf, -0.70) & V24=[-...",26167,100.0,2.190839,1
3,"V14=(-inf, -1.02) & V17=(-inf, -0.46) & V24=[-...",24487,100.0,2.190839,1
4,"V17=(-inf, -0.46) & V10=(-inf, -0.70) & V24=[-...",24454,100.0,2.190839,1
5,"V10=(-inf, -0.70) & V11=[0.91, inf) & V24=[-1....",24040,100.0,2.190839,1
6,"V17=(-inf, -0.46) & V11=[0.91, inf) & V24=[-1....",23680,100.0,2.190839,1
7,"V17=(-inf, -0.46) & V12=(-inf, -0.87) & V24=[-...",23188,100.0,2.190839,1
8,"V14=(-inf, -1.02) & V27=[0.45, 1.24)",22868,100.0,2.190839,1
9,"V14=(-inf, -1.02) & V9=(-inf, -1.03) & V22=[-0...",22677,100.0,2.190839,1


In [14]:
second_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(second_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Class
0    57.387088
1    42.612912
Name: proportion, dtype: float64

In [15]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"V14=(-inf, -1.02) & V27=[0.34, 1.27)",20528,100.0,2.346707,1
1,"V14=(-inf, -1.02) & V27=[0.34, 1.27) & V22=(-i...",20528,100.0,2.346707,1
2,"V12=(-inf, -0.71) & V3=[-1.00, -0.65) & V27=[0...",19351,100.0,2.346707,1
3,"V11=[0.98, inf) & V26=[0.17, 1.19)",18122,100.0,2.346707,1
4,"V11=[0.98, inf) & V26=[0.17, 1.19) & V22=(-inf...",18122,100.0,2.346707,1
5,"V11=[0.98, inf) & V27=[0.34, 1.27)",17749,100.0,2.346707,1
6,"V11=[0.98, inf) & V27=[0.34, 1.27) & V22=(-inf...",17749,100.0,2.346707,1
7,"V12=(-inf, -0.71) & V11=[0.98, inf) & V26=[0.1...",17226,100.0,2.346707,1
8,"V12=(-inf, -0.71) & V3=[-1.00, -0.65) & V15=[-...",16711,100.0,2.346707,1
9,"V14=(-inf, -1.02) & V4=[0.97, inf) & V3=[-1.00...",16575,100.0,2.346707,1


In [16]:
third_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(third_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Class
0    59.894836
1    40.105164
Name: proportion, dtype: float64

In [17]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"V14=(-inf, -0.76) & V17=(-inf, -0.34) & V26=[0...",18627,100.0,2.493444,1
1,"V14=(-inf, -0.76) & V12=(-inf, -0.71) & V26=[0...",18148,100.0,2.493444,1
2,"V17=(-inf, -0.34) & V3=[-0.95, -0.61) & V15=[-...",15838,100.0,2.493444,1
3,"V17=(-inf, -0.34) & V26=[0.16, 1.19) & V15=[-0...",15088,100.0,2.493444,1
4,"V12=(-inf, -0.71) & V3=[-0.95, -0.61) & V15=[-...",14961,100.0,2.493444,1
5,"V17=(-inf, -0.34) & V24=[-0.49, -0.21) & V22=[...",14390,100.0,2.493444,1
6,"V12=(-inf, -0.71) & V3=[-0.95, -0.61) & V1=[-1...",14257,100.0,2.493444,1
7,"V14=[-0.55, -0.35) & V17=(-inf, -0.34) & V22=[...",13542,100.0,2.493444,1
8,"V14=[-0.76, -0.55) & V7=[-0.35, -0.11)",13265,100.0,2.493444,1
9,"V17=(-inf, -0.34) & V3=[-0.95, -0.61) & V19=[1...",13116,100.0,2.493444,1


In [18]:
forth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(forth_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Class
0    62.326926
1    37.673074
Name: proportion, dtype: float64

In [19]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"V14=[-0.62, -0.53) & V17=(-inf, -0.33)",13643,100.0,2.654416,1
1,"V14=[-0.53, -0.29) & V17=(-inf, -0.33) & V22=[...",12462,100.0,2.654416,1
2,"V14=[-0.62, -0.53) & V17=(-inf, -0.33) & V22=[...",11198,100.0,2.654416,1
3,"V17=(-inf, -0.33) & V3=[-0.13, 0.03) & V22=[-0...",10533,100.0,2.654416,1
4,"V17=(-inf, -0.33) & V12=[-0.71, -0.57) & V25=[...",10358,100.0,2.654416,1
5,"V17=(-inf, -0.33) & V10=[-0.70, -0.47) & V15=[...",9670,100.0,2.654416,1
6,"V17=(-inf, -0.33) & V12=[-0.71, -0.57) & V13=(...",9608,100.0,2.654416,1
7,"V14=[-0.81, -0.62) & V17=(-inf, -0.33) & V13=[...",9426,100.0,2.654416,1
8,"V14=[-0.81, -0.62) & V17=(-inf, -0.33) & V4=[0...",9359,100.0,2.654416,1
9,"V17=(-inf, -0.33) & V4=[0.12, 0.43) & V13=[-0....",9161,100.0,2.654416,1


In [20]:
fifth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(fifth_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

Class
0    64.160468
1    35.839532
Name: proportion, dtype: float64

In [21]:
import numpy as np

In [22]:
def convert_interval_to_query(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a pandas .query() compatible string.
    """
    query_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column
        if col_conditions:
            query_conditions.append(" & ".join(col_conditions))
            
    # Combine all column rules into the final query string
    return " & ".join(query_conditions)

# --- Test the converter ---
input_string = first_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  V10=(-inf, -1.66) & V16=(-inf, -1.09)
Output: V10 < -1.66 & V16 < -1.09


In [23]:
# --- Test the converter ---
input_string = second_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  V14=(-inf, -1.02) & V12=(-inf, -0.87) & V24=[-1.46, -0.49)
Output: V14 < -1.02 & V12 < -0.87 & V24 >= -1.46 & V24 < -0.49


In [24]:
# --- Test the converter ---
input_string = third_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  V14=(-inf, -1.02) & V27=[0.34, 1.27)
Output: V14 < -1.02 & V27 >= 0.34 & V27 < 1.27


In [25]:
# --- Test the converter ---
input_string = forth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  V14=(-inf, -0.76) & V17=(-inf, -0.34) & V26=[0.16, 1.19)
Output: V14 < -0.76 & V17 < -0.34 & V26 >= 0.16 & V26 < 1.19


In [26]:
# --- Test the converter ---
input_string = fifth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  V14=[-0.62, -0.53) & V17=(-inf, -0.33)
Output: V14 >= -0.62 & V14 < -0.53 & V17 < -0.33


In [27]:
def convert_interval_to_sql(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a SQL WHERE clause compatible string.
    """
    sql_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column using SQL 'AND'
        if col_conditions:
            sql_conditions.append(" AND ".join(col_conditions))
            
    # Combine all column rules into the final SQL string using 'AND'
    # Parentheses are added around variables that have both an upper and lower bound
    return " AND ".join(f"({cond})" if "AND" in cond else cond for cond in sql_conditions)

# --- Test the converter ---
input_string = first_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  V10=(-inf, -1.66) & V16=(-inf, -1.09)
Output SQL WHERE clause: V10 < -1.66 AND V16 < -1.09


In [28]:
# --- Test the converter ---
input_string = second_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  V14=(-inf, -1.02) & V12=(-inf, -0.87) & V24=[-1.46, -0.49)
Output SQL WHERE clause: V14 < -1.02 AND V12 < -0.87 AND (V24 >= -1.46 AND V24 < -0.49)


In [29]:
# --- Test the converter ---
input_string = third_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  V14=(-inf, -1.02) & V27=[0.34, 1.27)
Output SQL WHERE clause: V14 < -1.02 AND (V27 >= 0.34 AND V27 < 1.27)


In [30]:
# --- Test the converter ---
input_string = forth_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  V14=(-inf, -0.76) & V17=(-inf, -0.34) & V26=[0.16, 1.19)
Output SQL WHERE clause: V14 < -0.76 AND V17 < -0.34 AND (V26 >= 0.16 AND V26 < 1.19)


In [31]:
# --- Test the converter ---
input_string = fifth_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  V14=[-0.62, -0.53) & V17=(-inf, -0.33)
Output SQL WHERE clause: (V14 >= -0.62 AND V14 < -0.53) AND V17 < -0.33


In [33]:
import duckdb
result = duckdb.sql("""

SELECT CASE WHEN V10 < -1.66 AND V16 < -1.09 THEN 1
WHEN V14 < -1.02 AND V12 < -0.87 AND (V24 >= -1.46 AND V24 < -0.49) THEN 2
WHEN V14 < -1.02 AND (V27 >= 0.34 AND V27 < 1.27) THEN 3
WHEN V14 < -0.76 AND V17 < -0.34 AND (V26 >= 0.16 AND V26 < 1.19) THEN 4
WHEN (V14 >= -0.62 AND V14 < -0.53) AND V17 < -0.33 THEN 5
ELSE 0 END AS SEGMENTS, 
COUNT(*) AS COUNTS,
SUM(Class) AS EVENT,
(SUM(Class) / COUNT(*)) * 100 AS RESPONSE_RATE,
FROM data
GROUP BY SEGMENTS

""").df()

result



,SEGMENTS,COUNTS,EVENT,RESPONSE_RATE
0,0,443128,158815.0,35.839532
1,1,45565,45564.0,99.997805
2,2,27633,27633.0,100.000000
3,3,20745,20744.0,99.995180
4,4,18523,18523.0,100.000000
5,5,13036,13036.0,100.000000
